# Importing all required libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Resolve project folders relative to this notebook (portable across computers)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT / "kbs-industrial-fault-diagnosis").exists():
    PROJECT_ROOT = PROJECT_ROOT / "kbs-industrial-fault-diagnosis"

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "docs" / "figures"
TABLES_DIR = PROJECT_ROOT / "docs" / "tables"
MODELS_DIR = PROJECT_ROOT / "models"

for folder in [PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Model Selection & Tuning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# Preprocessing & Pipeline Construction
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Model Families
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score


# Data Preprocessing 

In [ ]:
raw_file_path = RAW_DIR / "ai4i2020_messy.csv"
print("Loading dataset from:", raw_file_path)


In [ ]:
df=pd.read_csv(raw_file_path)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#checking null values 
df.isna().sum()

In [ ]:
#checking duplicates
print("number of duplicate rows:",df.duplicated().sum())
# Check duplicate UDI records
print("\nDuplicate UDI Records:")
print(df["UDI"].duplicated().sum())

# Expository Data Analytics

In [ ]:
# Select numerical columns
num_cols = df.select_dtypes(include='number').columns

# Exclude identifier and binary columns
num_cols = [
    
    col for col in num_cols
    if col != 'UDI' and df[col].nunique(dropna=True) > 2
]

print("Numerical features selected for outlier analysis:")
print(num_cols)

In [ ]:
# Create and save boxplots for numerical features
output_dir = FIGURES_DIR
output_dir.mkdir(parents=True, exist_ok=True)

for col in num_cols:
    plt.figure(figsize=(6, 2))
    sns.boxplot(x=df[col], color='skyblue')
    plt.title(f'Looking for Outliers in: {col}')

    safe_col_name = col.replace(" ", "_").replace("[", "").replace("]", "").replace("/", "_")
    file_path = output_dir / f"boxplot_{safe_col_name}.png"

    plt.savefig(file_path, bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()
print(f"All boxplots saved successfully in: {output_dir}")


In [ ]:
# Finding outliers in numerical features

for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    if len(outliers) > 0:
        print(f"{col}: Found {len(outliers)} outliers.")
    else:
        print(f"{col}: No outliers.")

In [ ]:
# target distribution plot
plt.figure(figsize=(6, 4))
sns.countplot(
    data=df,
    x='Machine failure',
    hue='Machine failure',
    palette=['#2ecc71', '#e74c3c'],
    legend=False,
)

plt.title('Machine Failure Count (0 = Normal, 1 = Failed)')
plt.xlabel('Machine Failure')
plt.ylabel('Count')

plt.savefig(
    os.path.join(output_dir, 'target_distribution.png'), bbox_inches='tight'
)
plt.show()
plt.close()

In [ ]:
#Distribution Histograms (One Graph per Feature)

# Plot a simple histogram for each feature
for col in num_cols:
    plt.figure(figsize=(6, 3))
    sns.histplot(df[col], kde=True, color='teal')
    plt.title(f'Distribution of {col}')
    
    # Save image with safe filename
    clean_name = col.replace(" ", "_").replace("[", "").replace("]", "")
    plt.savefig(os.path.join(output_dir, f"hist_{clean_name}.png"), bbox_inches='tight')
    plt.show()
    plt.close()

In [ ]:
plt.figure(figsize=(8, 6))
fet_to_check=num_cols+['Machine failure']
sns.heatmap(df[fet_to_check].corr(),annot=True,cmap="coolwarm",fmt=".2f")
plt.title('Feature Correlation Heatmap')
# Save and display image
plt.savefig(os.path.join(output_dir, "correlation_heatmap.png"), bbox_inches='tight')
plt.show()
plt.close()

print(f"All graphs saved in: {output_dir}")

# Data cleaning 

In [ ]:
# Remove only fully identical rows.
rows_before = len(df)
exact_duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

print("Rows before exact-duplicate removal:", rows_before)
print("Fully identical duplicate rows removed:", exact_duplicate_count)
print("Rows after exact-duplicate removal:", len(df))


In [ ]:
#checking duplicates
print("number of duplicate rows:",df.duplicated().sum())

In [ ]:
# Audit repeated UDI values before deciding how to handle them.
# Do NOT silently use groupby(...).first(): it can discard conflicting sensor
# readings or failure labels belonging to the same UDI.
duplicate_udi_mask = df["UDI"].duplicated(keep=False)
duplicate_udi_records = df.loc[duplicate_udi_mask].sort_values("UDI").copy()

print("Rows involved in repeated UDI values:", len(duplicate_udi_records))
print("Number of repeated UDI values:",
      duplicate_udi_records["UDI"].nunique())

# Repeated UDI rows may differ because (a) one copy has NaN where the other has a
# value, or (b) one copy carries an impossible sensor value (e.g. 15000 rpm, 500 Nm).
# Physically impossible readings are masked first, so only REAL conflicts remain.
plausible_ranges = {
    'Air temperature [K]': (273, 350), 'Process temperature [K]': (273, 350),
    'Rotational speed [rpm]': (0, 4000), 'Torque [Nm]': (0, 100),
    'Tool wear [min]': (0, 300)
}
def invalid_mask(frame):
    bad = pd.DataFrame(False, index=frame.index, columns=list(plausible_ranges))
    for col, (lo, hi) in plausible_ranges.items():
        bad[col] = (frame[col] <= lo) | (frame[col] > hi) if col in ['Rotational speed [rpm]', 'Torque [Nm]'] \
                   else (frame[col] < lo) | (frame[col] > hi)
    return bad

comparison_cols = [c for c in df.columns if c != "UDI"]
checked = duplicate_udi_records.copy()
for col, is_bad in invalid_mask(checked).items():
    checked.loc[is_bad, col] = np.nan
udi_conflict_summary = checked.groupby("UDI")[comparison_cols].nunique(dropna=True)
conflicting_udis = udi_conflict_summary.gt(1).any(axis=1)
print("Repeated UDIs with real conflicting values:", int(conflicting_udis.sum()))

# Save the full audit for review. Keep one row per UDI only when repeated
# records are exact duplicates in all non-UDI fields. Stop if conflicts exist.
duplicate_udi_records.to_csv(
    PROCESSED_DIR / "repeated_udi_audit.csv", index=False
)

if conflicting_udis.any():
    conflicting_ids = conflicting_udis[conflicting_udis].index.tolist()
    raise ValueError(
        "Conflicting records found for repeated UDI values. "
        "Review data/processed/repeated_udi_audit.csv before continuing. "
        f"Example UDI values: {conflicting_ids[:10]}"
    )

# Merge repeated UDI records: groupby().first() keeps the first NON-NULL value per
# column, so a missing value in one copy is filled from the other copy.
# Impossible sensor values are blanked first so the valid copy's value is kept.
for col, is_bad in invalid_mask(df).items():
    df.loc[is_bad & df["UDI"].duplicated(keep=False), col] = np.nan
df = df.groupby("UDI", as_index=False, sort=False).first()
print("Rows after merging repeated UDI records:", len(df))
print("Remaining duplicate UDI:", int(df["UDI"].duplicated().sum()))


In [ ]:
# Recover missing Type values using Product ID
missing_before = df['Type'].isnull().sum()

df.loc[df['Type'].isnull(), 'Type'] = (
    df.loc[df['Type'].isnull(), 'Product ID'].str[0]
)

missing_after = df['Type'].isnull().sum()

print("Missing Type values before recovery:", missing_before)
print("Missing Type values after recovery:", missing_after)

print("\nType distribution:")
print(df['Type'].value_counts(dropna=False))


In [ ]:
# STEP 1: Label Consistency Audit

failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# Check whether at least one failure mode is active
df['Failure_Mode_Indicated'] = (
    df[failure_modes].sum(axis=1) > 0
).astype(int)

# Compare with Machine failure
print("Machine Failure Distribution:")
print(df['Machine failure'].value_counts())

print("\nFailure Mode Indicated Distribution:")
print(df['Failure_Mode_Indicated'].value_counts())

# Find inconsistent records
inconsistent_records = df[
    df['Machine failure'] != df['Failure_Mode_Indicated']
]

print("\nInconsistent Records:", len(inconsistent_records))

print("\nInconsistent Record Details:")
display(
    inconsistent_records[
        ['Machine failure'] + failure_modes +
        ['Failure_Mode_Indicated']
    ].head(10)
)

In [ ]:
# STEP 2: Categorize inconsistent records

false_failure_modes = inconsistent_records[
    (inconsistent_records['Machine failure'] == 0) &
    (inconsistent_records['Failure_Mode_Indicated'] == 1)
]

missing_failure_modes = inconsistent_records[
    (inconsistent_records['Machine failure'] == 1) &
    (inconsistent_records['Failure_Mode_Indicated'] == 0)
]

print("Failure modes present but Machine failure = 0:",
      len(false_failure_modes))

print("Machine failure = 1 but no failure mode indicated:",
      len(missing_failure_modes))

In [ ]:
processed_dir = PROCESSED_DIR
audit_path = processed_dir / "inconsistent_label_records.csv"

inconsistent_records.to_csv(audit_path, index=False)

print("Saved at:", audit_path)
print("File exists:", audit_path.exists())


In [ ]:
# Remove audit helper column before ML split
df = df.drop(columns=['Failure_Mode_Indicated'])

print("Failure_Mode_Indicated removed:",
      'Failure_Mode_Indicated' not in df.columns)

In [ ]:
# Save cleaned dataset
file_name = "ai4i2020_cleaned.csv"
output_path = PROCESSED_DIR / file_name

df.to_csv(output_path, index=False)

print(f"Dataset successfully saved to:\n{output_path}")


Train-test split

In [ ]:
from sklearn.model_selection import train_test_split

# Separate input features and target variable
X = df.drop(columns=['Machine failure', 'UDI', 'Product ID'])  # identifiers are not features
y = df['Machine failure']

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Preserve failure-mode labels for KBS
failure_modes = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

y_train_modes = df.loc[X_train.index, failure_modes].copy()
y_test_modes = df.loc[X_test.index, failure_modes].copy()

# Remove failure-mode labels from ML features only
X_train = X_train.drop(columns=failure_modes)
X_test = X_test.drop(columns=failure_modes)

# Check dataset sizes
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

# Verify that leakage columns are removed
print("\nFailure_Mode_Indicated in X_train:",
      'Failure_Mode_Indicated' in X_train.columns)

print("\nFailure mode flags in X_train:")
print([col for col in failure_modes if col in X_train.columns])

# Verify preserved KBS labels
print("\nTraining KBS labels:", y_train_modes.shape)
print("Testing KBS labels:", y_test_modes.shape)

In [ ]:
# Export train and test sets for reproducibility
processed_dir = TABLES_DIR
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
X_test.to_csv(processed_dir / "X_test.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)

print("Train and test splits saved successfully!")
print(f"Saved directory: {processed_dir}\n")

files = ["X_train.csv", "y_train.csv", "X_test.csv", "y_test.csv"]
for file in files:
    file_path = processed_dir / file
    size_kb = file_path.stat().st_size / 1024
    print(f" - {file:<12}: {size_kb:.2f} KB")


Missing value handling

In [ ]:
# Check missing values before imputation
print("Missing values in training data:")
print(X_train.isnull().sum())

print("\nMissing values in testing data:")
print(X_test.isnull().sum())

# Numerical features: Fill missing values with training median
sensor_cols = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

for col in sensor_cols:
    if col in X_train.columns:
        col_median = X_train[col].median()

        X_train[col] = X_train[col].fillna(col_median)
        X_test[col] = X_test[col].fillna(col_median)

# Verify missing values
print("\nRemaining missing values in training data:",
      X_train.isnull().sum().sum())

print("Remaining missing values in testing data:",
      X_test.isnull().sum().sum())

Invalid values and IQR capping

In [ ]:
import numpy as np

# Define physically plausible sensor ranges.
# Boundaries for rotational speed and torque are exclusive at zero.
valid_ranges = {
    'Air temperature [K]':     (273, 350),
    'Process temperature [K]': (273, 350),
    'Rotational speed [rpm]':  (0, 4000),
    'Torque [Nm]':             (0, 100),
    'Tool wear [min]':         (0, 300)
}

exclusive_lower = ['Rotational speed [rpm]', 'Torque [Nm]']

# Preserve an audit of invalid values BEFORE replacing them.
invalid_audit_rows = []

print("Invalid values identified (Train / Test):")
for col, (lower, upper) in valid_ranges.items():
    if col in exclusive_lower:
        train_invalid = (X_train[col] <= lower) | (X_train[col] > upper)
        test_invalid = (X_test[col] <= lower) | (X_test[col] > upper)
    else:
        train_invalid = (X_train[col] < lower) | (X_train[col] > upper)
        test_invalid = (X_test[col] < lower) | (X_test[col] > upper)

    train_count = int(train_invalid.sum())
    test_count = int(test_invalid.sum())

    for split_name, frame, mask in [
        ("train", X_train, train_invalid),
        ("test", X_test, test_invalid)
    ]:
        if mask.any():
            audit_part = frame.loc[mask, [col]].copy()
            audit_part["feature"] = col
            audit_part["split"] = split_name
            audit_part["original_index"] = audit_part.index
            audit_part["original_value"] = audit_part[col]
            invalid_audit_rows.append(
                audit_part[["split", "original_index", "feature", "original_value"]]
            )

    X_train.loc[train_invalid, col] = np.nan
    X_test.loc[test_invalid, col] = np.nan
    print(f"{col}: {train_count} / {test_count}")

if invalid_audit_rows:
    invalid_audit = pd.concat(invalid_audit_rows, ignore_index=True)
else:
    invalid_audit = pd.DataFrame(
        columns=["split", "original_index", "feature", "original_value"]
    )

invalid_audit_path = PROCESSED_DIR / "invalid_sensor_values_audit.csv"
invalid_audit.to_csv(invalid_audit_path, index=False)
print("Invalid-reading audit saved to:", invalid_audit_path)

# Impute missing and invalid values using TRAINING medians only.
for col in valid_ranges:
    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

print("\nRemaining missing values:")
print("Training:", X_train.isnull().sum().sum())
print("Testing:", X_test.isnull().sum().sum())

print("\nFinal sensor ranges in training data:")
print(X_train[list(valid_ranges)].agg(['min', 'max']).round(2))
print("\nFinal sensor ranges in testing data:")
print(X_test[list(valid_ranges)].agg(['min', 'max']).round(2))
print("\nDomain-based invalid-value handling completed.")


In [ ]:
# Save processed training and testing datasets
processed_dir = TABLES_DIR
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train_cleaned.csv", index=False)
X_test.to_csv(processed_dir / "X_test_cleaned.csv", index=False)
y_train.to_csv(processed_dir / "y_train.csv", index=False)
y_test.to_csv(processed_dir / "y_test.csv", index=False)

print("Processed training and testing datasets saved successfully.")


Feature Engineering

In [ ]:
# STEP 1: Create physics-based features

def create_features(data):

    data = data.copy()

    # 1. Temperature Difference
    data['Temp_diff'] = (
        data['Process temperature [K]'] -
        data['Air temperature [K]']
    )

    # 2. Mechanical Power (Watts)
    data['Power'] = (
        data['Torque [Nm]'] *
        (data['Rotational speed [rpm]'] * (2 * np.pi / 60))
    )

    # 3. Overstrain Index
    data['Overstrain_Index'] = (
        data['Tool wear [min]'] *
        data['Torque [Nm]']
    )

    # 4. Torque-Speed Ratio
    data['Torque_Speed_Ratio'] = (
        data['Torque [Nm]'] /
        (data['Rotational speed [rpm]'] + 1e-5)
    )

    return data


X_train = create_features(X_train)
X_test = create_features(X_test)


# STEP 2: One-Hot Encoding of Product Type
X_train = pd.get_dummies(
    X_train, columns=['Type'], drop_first=True, dtype=int
)

X_test = pd.get_dummies(
    X_test, columns=['Type'], drop_first=True, dtype=int
)

# Ensure both datasets have identical columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)


# STEP 3: Verify engineered features
print("Training dataset shape:", X_train.shape)
print("Testing dataset shape:", X_test.shape)

print("\nEngineered features:")
print([
    'Temp_diff',
    'Power',
    'Overstrain_Index',
    'Torque_Speed_Ratio'
])

print("\nFinal training columns:")
print(X_train.columns.tolist())

print("\nTraining data preview:")
display(X_train.head())

In [ ]:
# Save engineered training and testing datasets
processed_dir = PROCESSED_DIR
processed_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(processed_dir / "X_train_engineered.csv", index=False)
X_test.to_csv(processed_dir / "X_test_engineered.csv", index=False)

print("Engineered training and testing datasets saved successfully.")


In [ ]:
# Check target distribution in training and testing data

print("TRAINING DATA DISTRIBUTION")
print(y_train.value_counts())
print("\nTraining percentage:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTESTING DATA DISTRIBUTION")
print(y_test.value_counts())
print("\nTesting percentage:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Feature Scaling Logistic Regression

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale features for Logistic Regression only
binary_cols = ['Type_L', 'Type_M']
continuous_cols = [
    col for col in X_train.columns
    if col not in binary_cols
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(
    X_train[continuous_cols]
)

X_test_scaled[continuous_cols] = scaler.transform(
    X_test[continuous_cols]
)

print("Feature scaling completed for Logistic Regression.")
print("Training shape:", X_train_scaled.shape)
print("Testing shape:", X_test_scaled.shape)

Model_training 


In [ ]:
from sklearn.linear_model import LogisticRegression

model_lg = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000
)

model_lg.fit(X_train_scaled, y_train)

print("Logistic Regression training completed.")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_estimators=200,
    max_depth=10,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed.")

In [ ]:
from xgboost import XGBClassifier

# Calculate class imbalance ratio
scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

# Create safe feature names for XGBoost
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

X_train_xgb.columns = [
    col.replace('[', '').replace(']', '').replace('<', '')
    .replace(' ', '_')
    for col in X_train_xgb.columns
]

X_test_xgb.columns = X_train_xgb.columns

# Create XGBoost model
model_xg = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

# Train XGBoost
model_xg.fit(X_train_xgb, y_train)

print("XGBoost training completed.")
print("Scale positive weight:", round(scale_pos_weight, 2))

In [ ]:
from lightgbm import LGBMClassifier
import re

# Create separate copies for LightGBM
X_train_lgbm = X_train.copy()
X_test_lgbm = X_test.copy()

# Clean feature names
def clean_col(col):
    return re.sub(r'[^A-Za-z0-9_]', '_', col)

X_train_lgbm.columns = [
    clean_col(col) for col in X_train_lgbm.columns
]

X_test_lgbm.columns = X_train_lgbm.columns

# Calculate class imbalance ratio
scale_pos_weight = (
    (y_train == 0).sum() /
    (y_train == 1).sum()
)

# Create LightGBM model
lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=-1,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbosity=-1
)

# Train LightGBM
lgbm_model.fit(X_train_lgbm, y_train)

print("LightGBM training completed.")
print("Scale positive weight:", round(scale_pos_weight, 2))

prediction and evaluation

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Logistic Regression
y_pred_lg = model_lg.predict(X_test_scaled)
y_proba_lg = model_lg.predict_proba(X_test_scaled)[:, 1]

print("LOGISTIC REGRESSION")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lg))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lg))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lg))


# Random Forest
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("\nRANDOM FOREST")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))


# XGBoost
y_pred_xgb = model_xg.predict(X_test_xgb)
y_proba_xgb = model_xg.predict_proba(X_test_xgb)[:, 1]

print("\nXGBOOST")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))

y_pred_lgbm = lgbm_model.predict(X_test_lgbm)
y_prob_lgbm = lgbm_model.predict_proba(X_test_lgbm)[:, 1]

print("LIGHTGBM EVALUATION")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgbm))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lgbm,
    target_names=['Normal', 'Failure'],
    zero_division=0
))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob_lgbm))

In [ ]:
# ROC-AUC values are already included in the model evaluation (cell above)
# and the baseline comparison table below. This separate printout was removed
# to avoid repeating the same metric without adding new analysis.


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

# Baseline model comparison (before hyperparameter tuning).
comparison_data = []

models_results = {
    "Logistic Regression": (y_pred_lg, y_proba_lg),
    "Random Forest": (y_pred_rf, y_prob_rf),
    "XGBoost": (y_pred_xgb, y_proba_xgb),
    "LightGBM": (y_pred_lgbm, y_prob_lgbm)
}

for name, (pred, prob) in models_results.items():
    cm = confusion_matrix(y_test, pred)
    comparison_data.append({
        "Model": name,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "False Positives": cm[0][1],
        "False Negatives": cm[1][0]
    })

comparison_df = pd.DataFrame(comparison_data)
table_dir = TABLES_DIR
table_dir.mkdir(parents=True, exist_ok=True)

csv_save_path = table_dir / "baseline_model_comparison.csv"
comparison_df.to_csv(csv_save_path, index=False)

display(comparison_df.round(4))
print("Baseline comparison table saved at:", csv_save_path)


In [ ]:
#this XGBoost model can perform well but we can increase acuracy even more using hyper parameter tunning

Hyper Parameter Tunning:

In [ ]:
import re

def clean_col(col):
    return re.sub(r'[\[\]<>]', '', col)

X_train_xgb = X_train_xgb.copy()
X_test_xgb = X_test_xgb.copy()

X_train_xgb.columns = [clean_col(c) for c in X_train_xgb.columns]
X_test_xgb.columns = [clean_col(c) for c in X_test_xgb.columns]

print("XGBoost feature names:")
print(X_train_xgb.columns.tolist())

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier

# Parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'scale_pos_weight': [10, 15, 20, 28.52]
}

# Base XGBoost model
base_xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    n_jobs=1
)

# 5-Fold Stratified Cross Validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Randomized Search
search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_grid,
    n_iter=15,
    scoring='f1',
    cv=skf,
    random_state=42,
    n_jobs=-1
)

print("Running Randomized Search CV...")

search.fit(X_train_xgb, y_train)

print("\nBest Parameters:")
print(search.best_params_)

print("\nBest F1 Score (CV):")
print(search.best_score_)

# Best tuned model
best_xgb = search.best_estimator_

### Class Imbalance, SMOTE and Decision Threshold Experiments\n\nThe dataset is imbalanced, so model selection is based on minority-class metrics (especially failure-class precision, recall and F1-score), not accuracy alone. SMOTE and alternative decision thresholds were explored separately during experimentation. Based on the experiments reported by the project team, SMOTE did not improve the selected model's performance, and the default decision threshold of 0.50 performed better under the team's selected evaluation criteria. Therefore, the final workflow retains the original training distribution and the model's default threshold.\n\n**Validation note:** Threshold selection must be based on training/validation data (or cross-validation), not optimized on the final test set. Keep the final test set untouched for the final unbiased evaluation. Add the exact SMOTE and threshold comparison metrics to the report only if the corresponding experiment outputs are available; this notebook archive did not contain those experiment result tables.\n

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Predictions
y_pred_best = best_xgb.predict(X_test_xgb)
y_proba_best = best_xgb.predict_proba(X_test_xgb)[:, 1]

print("TUNED XGBOOST EVALUATION")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_best,
    target_names=['Normal', 'Failure'],
    zero_division=0
))

print("\nROC-AUC:", roc_auc_score(y_test, y_proba_best))

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Final comparison includes the tuned XGBoost model.
all_results = {
    "Logistic Regression": (y_pred_lg, y_proba_lg),
    "Random Forest": (y_pred_rf, y_prob_rf),
    "XGBoost (Baseline)": (y_pred_xgb, y_proba_xgb),
    "XGBoost (Tuned)": (y_pred_best, y_proba_best),
    "LightGBM": (y_pred_lgbm, y_prob_lgbm)
}

final_comparison = []
for name, (pred, prob) in all_results.items():
    cm = confusion_matrix(y_test, pred)
    final_comparison.append({
        "Model": name,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob),
        "False Positives": cm[0][1],
        "False Negatives": cm[1][0]
    })

df_comparison = pd.DataFrame(final_comparison)
table_dir = TABLES_DIR
table_dir.mkdir(parents=True, exist_ok=True)

save_path = table_dir / "final_model_comparison.csv"
df_comparison.to_csv(save_path, index=False)

print("Final Model Comparison:")
display(df_comparison.round(4))
print("Final comparison table saved:", save_path)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

importances = best_xgb.feature_importances_
feature_names = X_train_xgb.columns

importance_df = (
    pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    })
    .sort_values(by='Importance', ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=importance_df,
    x='Importance',
    y='Feature',
    hue='Feature',
    legend=False,
    palette='viridis'
)
plt.title('Tuned XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature Name')
plt.tight_layout()

fig_dir = TABLES_DIR / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / 'xgb_tuned_feature_importance.png'

plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print("Feature importance chart saved:", fig_path)
display(importance_df)


saving model artifacts


In [ ]:
import joblib

models_dir = MODELS_DIR
models_dir.mkdir(parents=True, exist_ok=True)

# Save tuned XGBoost model and its feature-name order.
joblib.dump(best_xgb, models_dir / "best_xgb_tuned.joblib")
joblib.dump(list(X_train_xgb.columns), models_dir / "feature_names.joblib")

# Save Logistic Regression scaler separately.
joblib.dump(scaler, models_dir / "scaler.joblib")

print("All model artifacts saved successfully.")
print("Model directory:", models_dir)
